# Relaciones entre clases en UML

Bienvenido/a. En esta lección vas a distinguir los 4 tipos de relación entre clases que existen en UML, de la más débil a la más fuerte, y a reconocerlos en código real.

## Objetivos
- Distinguir dependencia, asociación, agregación y composición.
- Reconocer cada relación mirando el código Python (qué se guarda como atributo vs. qué solo se recibe como parámetro).
- Saber cuándo una relación es débil (la parte sobrevive al todo) o fuerte (la parte muere con el todo).

---

**Ejemplo de la vida real:** piensa en cuánto depende una cosa de otra para existir:
- Pides un taxi (lo *usas* un momento: dependencia).
- Tienes un teléfono guardado en tu directorio (lo *tienes* de forma permanente, pero existe con independencia de ti: asociación).
- Un equipo de fútbol tiene jugadores, pero si el equipo se disuelve, los jugadores siguen jugando en otro lado (agregación).
- Tu corazón es parte de ti: no existe fuera de tu cuerpo (composición).

## Explicación detallada

| Relación | Fuerza | Cómo se ve en el código | Notación UML |
|---|---|---|---|
| Dependencia | La más débil | La clase aparece solo como **parámetro de un método**, nunca como atributo | Flecha punteada `- - >` |
| Asociación | Media | La clase se guarda como **atributo** en `__init__`, de forma permanente | Flecha continua `—>` |
| Agregación | Fuerte, pero la parte sobrevive | El objeto "parte" **ya existía** antes y se agrega a una colección del "todo" | Rombo vacío `◇—` del lado del todo |
| Composición | La más fuerte | El objeto "parte" se **crea dentro de** `__init__` del "todo", nadie más lo referencia | Rombo lleno `◆—` del lado del todo |

La pregunta clave para distinguir agregación de composición: **si destruyo el "todo", ¿la "parte" sigue teniendo sentido por sí sola?** Si sí (el jugador sigue jugando en otro equipo) es agregación. Si no (el motor de ese carro específico ya no sirve para nada) es composición.

## Ejemplo práctico 1 — Dependencia y Asociación

Dominio `Course`-`Student`-`Professor` (el mismo de `ejemplo_uml.py`).

In [1]:
class Course:
    def get_knowledge(self) -> str:
        return "knowledge"

class Student:
    def remember(self, knowledge: str) -> None:
        print(f"El estudiante ha recordado {knowledge}")

class Professor:
    def __init__(self, student: Student) -> None:
        # 'student' se guarda como atributo -> ASOCIACIÓN (permanente)
        self.student = student

    def teach(self, course: Course) -> None:
        # 'course' solo llega como parámetro, no se guarda -> DEPENDENCIA (momentanea)
        self.student.remember(course.get_knowledge())

student = Student()
course = Course()
professor = Professor(student)
professor.teach(course)
print("¿Professor guarda a Student?", hasattr(professor, "student"))
print("¿Professor guarda a Course?", hasattr(professor, "course"))

El estudiante ha recordado knowledge
¿Professor guarda a Student? True
¿Professor guarda a Course? False


## Ejemplo práctico 2 — Agregación (Equipo — Jugador)

El `Jugador` existe **antes** de ser fichado, y sigue existiendo si el equipo se disuelve.

In [2]:
class Jugador:
    def __init__(self, nombre: str) -> None:
        self.nombre = nombre

class Equipo:
    def __init__(self, nombre: str) -> None:
        self.nombre = nombre
        self.jugadores: list[Jugador] = []  # AGREGACIÓN: guarda referencias, no crea jugadores

    def fichar(self, jugador: Jugador) -> None:
        self.jugadores.append(jugador)

    def disolver(self) -> None:
        self.jugadores = []  # los jugadores NO desaparecen, solo dejan de estar aquí

equipo = Equipo("Tiburones")
ana = Jugador("Ana")
luis = Jugador("Luis")
equipo.fichar(ana)
equipo.fichar(luis)
print(f"{equipo.nombre} tiene a {[j.nombre for j in equipo.jugadores]}")
equipo.disolver()
print(f"Equipo disuelto. ¿Ana sigue existiendo?: {ana.nombre}")

Tiburones tiene a ['Ana', 'Luis']
Equipo disuelto. ¿Ana sigue existiendo?: Ana


## Ejemplo práctico 3 — Composición (Carro — Motor)

El `Motor` se crea **dentro de** `Carro.__init__` — nadie más tiene esa instancia.

In [3]:
class Motor:
    def __init__(self, caballos_de_fuerza: int) -> None:
        self.caballos_de_fuerza = caballos_de_fuerza

class Carro:
    def __init__(self, modelo: str, caballos_de_fuerza: int) -> None:
        self.modelo = modelo
        self.motor = Motor(caballos_de_fuerza)  # COMPOSICIÓN: se crea aquí, no se recibe de afuera

    def destruir(self) -> None:
        self.motor = None  # ESTE motor deja de existir junto con el carro

carro = Carro("Corolla", 140)
print(f"{carro.modelo} tiene un motor de {carro.motor.caballos_de_fuerza} hp")
carro.destruir()
print(f"Carro destruido. ¿Motor accesible?: {carro.motor}")

Corolla tiene un motor de 140 hp
Carro destruido. ¿Motor accesible?: None


## Ejercicios prácticos y preguntas de reflexión

1. Agrega una clase `Entrenador` que sea **asociación** de `Equipo` (el equipo lo guarda como atributo, pero el entrenador podría dirigir otro equipo después).
2. Agrega una clase `Rueda` que sea **composición** de `Carro` (4 ruedas creadas junto con el carro en `__init__`).
3. Para cada una de las 4 relaciones de este notebook, escribe en una frase qué pasaría si el "todo" se destruye.

### Autoevaluación
- ¿Cómo distingues en código Python una dependencia de una asociación, sin ver ningún diagrama?
- ¿Por qué `self.jugadores: list[Jugador] = []` en `Equipo` es agregación y no composición?
- Da un ejemplo propio (no visto en clase) de composición.

## Referencias y recursos
- [Documentación oficial de UML](https://www.uml.org/)
- [Refactoring Guru — UML class diagram relationships](https://refactoring.guru/es/design-patterns/uml)
- Ver también `class_uml/ejemplo_uml.py` (código fuente completo, ejecutable) y `class_uml/uml_diagrama_clases.ipynb`.